# CIFAR-10 KNN Classification: L1 vs L2

이 노트북은 CIFAR-10 데이터셋을 사용하여 KNN 분류기를 직접 구현하고, 거리 계산 방식으로 L1(Manhattan distance)과 L2(Euclidean distance)를 비교한다.

실행 시간이 너무 길어지는 것을 막기 위해 학습/테스트 데이터 일부만 사용한다. `TRAIN_LIMIT`, `TEST_LIMIT`, `K_VALUES` 값을 바꾸면 더 큰 실험도 가능하다.

In [ ]:
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')

RESULT_DIR = Path('results')
RESULT_DIR.mkdir(exist_ok=True)

CIFAR10_LABELS = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

## 1. 데이터 불러오기 및 전처리

In [ ]:
try:
    from tensorflow.keras.datasets import cifar10
except ImportError as exc:
    raise ImportError(
        'CIFAR-10을 자동으로 불러오려면 tensorflow가 필요합니다. '
        '설치 명령: pip install tensorflow'
    ) from exc

(x_train, y_train), (x_test, y_test) = cifar10.load_data()
y_train = y_train.ravel()
y_test = y_test.ravel()

print('Train:', x_train.shape, y_train.shape)
print('Test :', x_test.shape, y_test.shape)

In [ ]:
TRAIN_LIMIT = 5000
TEST_LIMIT = 1000
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)
train_idx = rng.choice(len(x_train), size=TRAIN_LIMIT, replace=False)
test_idx = rng.choice(len(x_test), size=TEST_LIMIT, replace=False)

x_train_small = x_train[train_idx]
y_train_small = y_train[train_idx]
x_test_small = x_test[test_idx]
y_test_small = y_test[test_idx]

# KNN은 픽셀 벡터 사이의 거리를 계산하므로 32x32x3 이미지를 3072차원 벡터로 변환한다.
X_train = x_train_small.reshape(TRAIN_LIMIT, -1).astype(np.float32) / 255.0
X_test = x_test_small.reshape(TEST_LIMIT, -1).astype(np.float32) / 255.0

print('Flattened train:', X_train.shape)
print('Flattened test :', X_test.shape)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, image, label in zip(axes.ravel(), x_train_small[:10], y_train_small[:10]):
    ax.imshow(image)
    ax.set_title(CIFAR10_LABELS[label])
    ax.axis('off')
fig.suptitle('CIFAR-10 Samples')
fig.tight_layout()
fig.savefig(RESULT_DIR / '5_cifar10_samples.png', dpi=150)
plt.show()

## 2. KNN 함수 구현

In [ ]:
def vote_labels(neighbor_labels, num_classes=10):
    predictions = np.empty(neighbor_labels.shape[0], dtype=np.int64)
    for i, labels in enumerate(neighbor_labels):
        counts = np.bincount(labels, minlength=num_classes)
        predictions[i] = counts.argmax()
    return predictions


def l2_distances(test_batch, train_data):
    test_sq = np.sum(test_batch * test_batch, axis=1, keepdims=True)
    train_sq = np.sum(train_data * train_data, axis=1)
    distances_sq = test_sq + train_sq - 2.0 * test_batch @ train_data.T
    return np.sqrt(np.maximum(distances_sq, 0.0))


def l1_distances(test_batch, train_data, train_chunk_size=500):
    distances = np.empty((len(test_batch), len(train_data)), dtype=np.float32)
    for start in range(0, len(train_data), train_chunk_size):
        end = min(start + train_chunk_size, len(train_data))
        distances[:, start:end] = np.abs(
            test_batch[:, None, :] - train_data[None, start:end, :]
        ).sum(axis=2)
    return distances


def knn_predict_many_k(train_data, train_labels, test_data, k_values, metric='l2', batch_size=50):
    max_k = max(k_values)
    predictions_by_k = {k: [] for k in k_values}
    for start in range(0, len(test_data), batch_size):
        end = min(start + batch_size, len(test_data))
        test_batch = test_data[start:end]

        if metric == 'l2':
            distances = l2_distances(test_batch, train_data)
        elif metric == 'l1':
            distances = l1_distances(test_batch, train_data)
        else:
            raise ValueError("metric must be 'l1' or 'l2'")

        neighbor_idx = np.argpartition(distances, kth=max_k - 1, axis=1)[:, :max_k]
        sorted_order = np.take_along_axis(distances, neighbor_idx, axis=1).argsort(axis=1)
        neighbor_idx = np.take_along_axis(neighbor_idx, sorted_order, axis=1)
        neighbor_labels = train_labels[neighbor_idx]

        for k in k_values:
            predictions_by_k[k].append(vote_labels(neighbor_labels[:, :k]))

    return {k: np.concatenate(predictions) for k, predictions in predictions_by_k.items()}

## 3. L1, L2 거리 방식 비교

In [ ]:
K_VALUES = [1, 3, 5, 7]
METRICS = ['l1', 'l2']

results = []

for metric in METRICS:
    start_time = perf_counter()
    predictions_by_k = knn_predict_many_k(
        X_train, y_train_small, X_test, k_values=K_VALUES, metric=metric, batch_size=50
    )
    elapsed = perf_counter() - start_time

    for k, y_pred in predictions_by_k.items():
        accuracy = np.mean(y_pred == y_test_small)
        results.append({
            'metric': metric.upper(),
            'k': k,
            'accuracy': accuracy,
            'time_sec': elapsed,
        })
        print(f"{metric.upper()} / k={k}: accuracy={accuracy:.4f}, metric_time={elapsed:.2f}s")

In [ ]:
try:
    import pandas as pd

    result_df = pd.DataFrame(results)
    display(result_df)
except ImportError:
    result_df = results
    for row in results:
        print(row)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for metric in METRICS:
    metric_name = metric.upper()
    metric_results = [row for row in results if row['metric'] == metric_name]
    ax.plot(
        [row['k'] for row in metric_results],
        [row['accuracy'] for row in metric_results],
        marker='o',
        linewidth=2,
        label=metric_name,
    )

ax.set_title('CIFAR-10 KNN Accuracy: L1 vs L2')
ax.set_xlabel('k')
ax.set_ylabel('Accuracy')
ax.set_xticks(K_VALUES)
ax.set_ylim(0, max(row['accuracy'] for row in results) + 0.05)
ax.legend(title='Distance')
fig.tight_layout()
fig.savefig(RESULT_DIR / '5_cifar10_knn_l1_l2_accuracy.png', dpi=150)
plt.show()

## 4. KNN 결정 영역 시각화

CIFAR-10 이미지는 3072차원 픽셀 벡터이므로 원래 공간의 KNN 영역을 직접 볼 수 없다. 따라서 학습 데이터를 PCA로 2차원에 투영한 뒤, 그 2차원 평면에서 KNN 결정 영역을 그린다.

In [ ]:
REGION_TRAIN_LIMIT = 1200
REGION_SCATTER_LIMIT = 350
REGION_K = 5
GRID_SIZE = 160

X_region = X_train[:REGION_TRAIN_LIMIT]
y_region = y_train_small[:REGION_TRAIN_LIMIT]

X_mean = X_region.mean(axis=0, keepdims=True)
X_centered = X_region - X_mean

# SVD를 이용해 PCA 2차원 좌표를 만든다.
_, _, vt = np.linalg.svd(X_centered, full_matrices=False)
components = vt[:2]
X_region_2d = X_centered @ components.T

x_min, x_max = X_region_2d[:, 0].min() - 1.0, X_region_2d[:, 0].max() + 1.0
y_min, y_max = X_region_2d[:, 1].min() - 1.0, X_region_2d[:, 1].max() + 1.0
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, GRID_SIZE),
    np.linspace(y_min, y_max, GRID_SIZE),
)
grid_points = np.c_[xx.ravel(), yy.ravel()].astype(np.float32)

print('PCA train points:', X_region_2d.shape)
print('Grid points:', grid_points.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)
cmap = plt.get_cmap('tab10')
levels = np.arange(11) - 0.5

for ax, metric in zip(axes, ['l1', 'l2']):
    region_pred = knn_predict_many_k(
        X_region_2d.astype(np.float32),
        y_region,
        grid_points,
        k_values=[REGION_K],
        metric=metric,
        batch_size=1000,
    )[REGION_K]
    region_map = region_pred.reshape(xx.shape)

    ax.contourf(xx, yy, region_map, levels=levels, cmap=cmap, alpha=0.28)
    scatter = ax.scatter(
        X_region_2d[:REGION_SCATTER_LIMIT, 0],
        X_region_2d[:REGION_SCATTER_LIMIT, 1],
        c=y_region[:REGION_SCATTER_LIMIT],
        cmap=cmap,
        s=18,
        edgecolors='k',
        linewidths=0.25,
    )
    ax.set_title(f'{metric.upper()} Decision Regions (k={REGION_K})')
    ax.set_xlabel('PCA component 1')
    ax.set_ylabel('PCA component 2')

cbar = fig.colorbar(scatter, ax=axes, ticks=np.arange(10), fraction=0.025, pad=0.02)
cbar.ax.set_yticklabels(CIFAR10_LABELS)
fig.suptitle('KNN Decision Regions on 2D PCA Projection')
fig.tight_layout()
fig.savefig(RESULT_DIR / '5_cifar10_knn_decision_regions.png', dpi=150)
plt.show()

## 5. 분석

- L1 거리는 각 픽셀 차이의 절댓값 합을 사용하므로 픽셀 단위 변화량을 직접 누적한다.
- L2 거리는 차이를 제곱한 뒤 합산하므로 큰 차이를 더 강하게 반영한다.
- CIFAR-10처럼 배경, 위치, 색상 변화가 큰 이미지에서는 단순 픽셀 기반 KNN의 정확도가 높지 않다.
- `k=1`은 가까운 한 장의 이미지에 크게 의존하고, 더 큰 `k`는 여러 이웃의 투표를 사용해 예측이 조금 더 안정적일 수 있다.
- 최종 비교는 위 표와 그래프의 accuracy 값을 기준으로 L1과 L2 중 어떤 거리 방식이 더 좋은지 판단한다.
- 결정 영역 그림은 PCA로 줄인 2차원 공간에서의 시각화이므로, 원본 3072차원 KNN 분류 결과와 완전히 같지는 않지만 L1/L2가 만드는 분류 경계의 차이를 직관적으로 보여준다.